## 🎯 Learning Objectives
* Implement a complete RAG pipeline from scratch using local components.
* Understand the interplay between document loading, chunking, embedding, vector storage, retrieval, and LLM prompting.
* Gain practical experience with in-memory vector search and local embedding models.


## RAG01-L09: Exercise - Build a Local RAG System from Scratch

### Task Description

In this exercise, you will build a simplified Retrieval-Augmented Generation (RAG) system entirely from scratch, using only local, in-memory components. The goal is to understand the fundamental steps involved in a RAG pipeline by implementing each part yourself, rather than relying on high-level libraries that abstract away the details.

You will be provided with a small set of text documents. Your RAG system should be able to answer questions based *only* on the information contained within these documents.

### Requirements

1.  **Document Loading & Chunking**: Load the provided raw text documents and split them into smaller, manageable chunks.
2.  **Embedding Generation**: Generate numerical vector embeddings for each document chunk using a local embedding model.
3.  **In-Memory Vector Store**: Create a simple in-memory data structure (e.g., a list of dictionaries or a custom class) to store the document chunks along with their corresponding embeddings.
4.  **Retrieval**: Implement a function that, given a user query, generates an embedding for the query and then performs a similarity search against your in-memory vector store to retrieve the top `k` most relevant document chunks.
5.  **Context Generation & LLM Simulation**: Construct a prompt that includes the retrieved chunks as context. Simulate an LLM's response by simply concatenating the query with the retrieved context, demonstrating how a real LLM would use this information.

### Evaluation Criteria

*   **Correctness**: Each component (chunking, embedding, vector store, retrieval, context generation) must be implemented correctly and function as described.
*   **Readability**: Code should be clean, well-structured, and include comments explaining key decisions and complex parts.
*   **Functionality**: The system should be able to process a query, retrieve relevant information, and present it in a structured manner.
*   **Efficiency (Basic)**: While not a performance-critical exercise, avoid obviously inefficient operations for small datasets.
*   **Adherence to "From Scratch"**: Minimize reliance on high-level RAG framework abstractions. Use basic Python data structures and libraries for core operations (e.g., `numpy` for vector math, `sentence-transformers` for embeddings).

Let's get started!

---

### Setup Code

Below, you'll find some initial setup code, including a mock dataset and necessary library imports. You are free to add more helper functions or modify this setup as needed for your solution.


In [ ]:
# Install necessary libraries (if not already installed)
# !pip install sentence-transformers numpy scikit-learn

import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# --- Mock Dataset ---
# A small collection of documents about Agentic AI and RAG systems.
raw_documents = [
    "Agentic AI refers to AI systems capable of autonomous decision-making and goal-oriented actions. They often involve planning, memory, and tool use.",
    "Retrieval-Augmented Generation (RAG) is an AI framework that enhances the factual accuracy and relevance of large language models (LLMs) by retrieving information from external knowledge bases.",
    "The core components of a RAG system include a document loader, a chunking mechanism, an embedding model, a vector database, a retriever, and an LLM for generation.",
    "Vector databases are specialized databases designed to store, manage, and search high-dimensional vector embeddings efficiently. They are crucial for RAG systems.",
    "An embedding model converts text into numerical vectors (embeddings) that capture semantic meaning. Texts with similar meanings will have embeddings that are close to each other in the vector space.",
    "Tool use in agentic AI allows agents to interact with external systems, APIs, or databases to gather information or perform actions beyond their inherent capabilities.",
    "Evaluation of RAG systems typically involves metrics like context relevance, answer faithfulness, and answer relevance. Human evaluation is often combined with automated metrics.",
    "Chunking strategies for RAG include fixed-size chunking, recursive character text splitting, and semantic chunking. The choice depends on the document structure and retrieval needs.",
    "LLM-01 is a prerequisite for this course, covering foundational concepts of Large Language Models, including their architecture, training, and basic prompting techniques."
]

# --- Global Parameters ---
CHUNK_SIZE = 200  # Max characters per chunk
CHUNK_OVERLAP = 50 # Overlap between chunks
TOP_K_RETRIEVAL = 3 # Number of top relevant chunks to retrieve

# --- Initialize Embedding Model ---
# Using a lightweight, local-first sentence transformer model.
# This model will be downloaded the first time it's used.
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded.")

# --- Helper Function: Simple Text Splitter ---
# This is a basic character-based splitter. For production, more sophisticated
# splitters (e.g., recursive character text splitter) are often used.
def simple_char_splitter(text: str, chunk_size: int, chunk_overlap: int) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        if end >= len(text):
            break
        start += chunk_size - chunk_overlap
    return chunks

print("Setup complete. You can now proceed with implementing your RAG system.")


### Your Implementation

Now it's your turn! Implement the RAG pipeline components as described in the requirements. Use the `raw_documents`, `embedding_model`, `simple_char_splitter`, `CHUNK_SIZE`, `CHUNK_OVERLAP`, and `TOP_K_RETRIEVAL` variables provided in the setup.

Your solution should include:

1.  **Document Processing**: Loop through `raw_documents`, split each into chunks, and store them.
2.  **Embedding Generation**: For each chunk, generate its embedding.
3.  **Vector Store**: Populate an in-memory list of dictionaries, where each dictionary contains at least the `chunk_text` and its `embedding`.
4.  **Retrieval Function**: A function `retrieve_chunks(query: str, vector_store: list, k: int) -> list[str]` that returns the top `k` relevant chunk texts.
5.  **RAG Query Function**: A function `ask_rag(query: str, vector_store: list, k: int) -> str` that orchestrates the retrieval and simulates the LLM response.

Feel free to add more helper functions as you deem necessary. Once implemented, test your `ask_rag` function with a few example queries.


In [ ]:
# --- Reference Solution ---

# 1. Document Processing & 2. Embedding Generation & 3. In-Memory Vector Store

# Our in-memory vector store will be a list of dictionaries.
# Each dictionary will contain the chunk text and its embedding.
vector_store = []

print("Processing documents and building vector store...")
for doc_id, doc_text in enumerate(raw_documents):
    # Split the document into chunks
    chunks = simple_char_splitter(doc_text, CHUNK_SIZE, CHUNK_OVERLAP)
    for chunk_id, chunk_text in enumerate(chunks):
        # Generate embedding for each chunk
        chunk_embedding = embedding_model.encode(chunk_text)
        
        # Store chunk text and its embedding in the vector store
        vector_store.append({
            "doc_id": doc_id,
            "chunk_id": chunk_id,
            "text": chunk_text,
            "embedding": chunk_embedding
        })

print(f"Vector store built with {len(vector_store)} chunks.")
# print(f"Example chunk: {vector_store[0]['text']}")
# print(f"Example embedding shape: {vector_store[0]['embedding'].shape}")

# 4. Retrieval Function
def retrieve_chunks(query: str, vector_store: list, k: int) -> list[str]:
    """
    Retrieves the top k most relevant chunks from the vector store for a given query.
    """
    # Generate embedding for the query
    query_embedding = embedding_model.encode(query)
    
    # Extract all chunk embeddings from the vector store
    all_chunk_embeddings = np.array([item["embedding"] for item in vector_store])
    
    # Calculate cosine similarity between query embedding and all chunk embeddings
    # Reshape query_embedding for sklearn's cosine_similarity to work correctly
    similarities = cosine_similarity(query_embedding.reshape(1, -1), all_chunk_embeddings)[0]
    
    # Get the indices of the top k most similar chunks
    top_k_indices = np.argsort(similarities)[::-1][:k]
    
    # Retrieve the actual chunk texts using these indices
    retrieved_chunks = [vector_store[i]["text"] for i in top_k_indices]
    
    return retrieved_chunks

# 5. RAG Query Function (with LLM Simulation)
def ask_rag(query: str, vector_store: list, k: int = TOP_K_RETRIEVAL) -> str:
    """
    Orchestrates the RAG process: retrieves chunks and simulates an LLM response.
    """
    print(f"\n--- Processing Query: '{query}' ---")
    
    # Step 1: Retrieve relevant chunks
    retrieved_context_chunks = retrieve_chunks(query, vector_store, k)
    
    print(f"Retrieved {len(retrieved_context_chunks)} chunks.")
    # print("Retrieved chunks:")
    # for i, chunk in enumerate(retrieved_context_chunks):
    #     print(f"  Chunk {i+1}: {chunk}")

    # Step 2: Construct context for the LLM
    # For a real LLM, this would be part of the prompt.
    context_str = "\n".join([f"- {chunk}" for chunk in retrieved_context_chunks])
    
    # Step 3: Simulate LLM response
    # In a real RAG system, you would send this prompt to an actual LLM.
    # For this exercise, we'll just show how the context is used.
    
    # Example of a prompt template for a real LLM:
    # prompt = f"""You are an AI assistant. Use the following context to answer the question.
    # If the answer is not in the context, state that you don't know.
    # Context:
    # {context_str}
    # Question: {query}
    # Answer:"""
    
    # Simulated LLM response:
    simulated_response = (
        f"Based on the retrieved information, here's what I found related to your query:\n\n"
        f"Query: {query}\n\n"
        f"Retrieved Context:\n{context_str}\n\n"
        f"(Note: In a real RAG system, an LLM would synthesize an answer from this context.)"
    )
    
    return simulated_response

# --- Test the RAG System ---

# Example Queries
queries = [
    "What are the main components of a RAG system?",
    "How do agentic AI systems make decisions?",
    "What is the purpose of an embedding model?",
    "Tell me about the prerequisites for this course.",
    "What are vector databases used for?",
    "What is the capital of France?" # This query should not find relevant info
]

for q in queries:
    response = ask_rag(q, vector_store)
    print(response)
    print("\n" + "="*80 + "\n")

print("Exercise complete. You have successfully built a local RAG system from scratch!")
